In [10]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression,Lasso,Ridge
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV

In [2]:
df = pd.read_csv('cardekho_imputated.csv')
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


## DATA CLEANING

In [3]:
df.drop(columns=['car_name','brand','Unnamed: 0'],axis=1,inplace=True)

In [4]:
num_features = [feature for feature in df.columns if df[feature].dtypes != 'object']
cat_features = [feature for feature in df.columns if df[feature].dtypes == 'object']
print(num_features)
print(cat_features)

['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']
['model', 'seller_type', 'fuel_type', 'transmission_type']


## FEATURE ENCODING AND SCALING

In [5]:
X = df.drop(columns=['selling_price'],axis=1)
y = df['selling_price']

le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])

In [6]:
nume_featires = X.select_dtypes(exclude=['object']).columns
ohe_columns = ['seller_type','fuel_type','transmission_type']

num_transofrmer = StandardScaler()
ohe_transformer = OneHotEncoder(handle_unknown='ignore',drop='first')

preprocessor = ColumnTransformer(
    [
        ('StandardScaler',num_transofrmer,nume_featires),
        ('OneHotEncoder',ohe_transformer,ohe_columns)
    ],remainder='passthrough'
)

In [7]:
X = preprocessor.fit_transform(X)
X

array([[-1.51971354e+00,  9.83561835e-01,  1.24733473e+00, ...,
         0.00000000e+00,  1.00000000e+00,  1.00000000e+00],
       [-2.25693398e-01, -3.43933310e-01, -6.90016231e-01, ...,
         0.00000000e+00,  1.00000000e+00,  1.00000000e+00],
       [ 1.53637659e+00,  1.64730941e+00,  8.49241548e-02, ...,
         0.00000000e+00,  1.00000000e+00,  1.00000000e+00],
       ...,
       [ 4.07550503e-01, -1.20595237e-02,  2.20538722e-01, ...,
         0.00000000e+00,  0.00000000e+00,  1.00000000e+00],
       [ 1.42624721e+00, -3.43933310e-01,  7.25418502e+01, ...,
         0.00000000e+00,  0.00000000e+00,  1.00000000e+00],
       [-1.02413136e+00, -1.33955467e+00, -8.25630799e-01, ...,
         0.00000000e+00,  1.00000000e+00,  0.00000000e+00]])

In [8]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## MODEL TRAINING

In [9]:
def evaluate_model(true,predicted):
    mae = mean_absolute_error(true,predicted)
    mse = mean_squared_error(true,predicted)
    r2 = r2_score(true,predicted)
    return mae,mse,r2

In [11]:
models = {
    "Random Forest":RandomForestRegressor(),
    "Adaboost":AdaBoostRegressor(),
    "KNN":KNeighborsRegressor(),
    "Decision Tree":DecisionTreeRegressor(),
    "Lasso":Lasso(),
    "Ridge":Ridge(),
    "Logistic Regression":LogisticRegression()
}
l=[]
for model_name,model_instance in models.items():
    model_instance.fit(X_train,y_train)
    y_train_pred = model_instance.predict(X_train)
    y_pred = model_instance.predict(X_test)
    model_train_mae,model_train_mse,model_train_r2 = evaluate_model(y_train,y_train_pred)
    model_test_mae,model_test_mse,model_test_r2 = evaluate_model(y_test,y_pred)
    print(model_name)
    print('Model training Performance')
    print("Root mean squared error: ",model_train_mse)
    print("Mean absolute error: ",model_train_mae)
    print("R2 score: ",model_train_r2)
    print('Model testing Performance')
    print("Root mean squared error: ",model_test_mse)
    print("Mean absolute error: ",model_test_mae)
    print("R2 score: ",model_test_r2)
    print('------------------------------------')
    l.append([model_train_r2,model_test_r2])

Random Forest
Model training Performance
Root mean squared error:  18814203554.645065
Mean absolute error:  40088.56436905486
R2 score:  0.9768022236780309
Model testing Performance
Root mean squared error:  51928425924.29766
Mean absolute error:  101854.66684424703
R2 score:  0.9310179568889878
------------------------------------
Adaboost
Model training Performance
Root mean squared error:  196996557644.591
Mean absolute error:  335275.2599774126
R2 score:  0.7571046753499777
Model testing Performance
Root mean squared error:  227011350426.03616
Mean absolute error:  355213.8498075199
R2 score:  0.6984367139376231
------------------------------------
KNN
Model training Performance
Root mean squared error:  106198752384.81505
Mean absolute error:  91396.54445165477
R2 score:  0.869057709706405
Model testing Performance
Root mean squared error:  64079155016.42069
Mean absolute error:  112578.24359390205
R2 score:  0.9148768529917701
------------------------------------
Decision Tree
Mo

In [21]:
adaboost_params = {
    'n_estimators':[10,20,40,50,100],
    'learning_rate':[0.1,0.2,0.4,0.5,1.0],
    'loss':['linear','square','exponential'],
    'random_state':[7,21,42]
}

In [22]:
ada_grid = GridSearchCV(estimator=AdaBoostRegressor(),param_grid=adaboost_params,cv=5,n_jobs=-1)
ada_random = RandomizedSearchCV(estimator=AdaBoostRegressor(),param_distributions=adaboost_params,cv=5,n_jobs=-1)
ada_grid.fit(X_train,y_train)
ada_random.fit(X_train,y_train)

RandomizedSearchCV(cv=5, estimator=AdaBoostRegressor(), n_jobs=-1,
                   param_distributions={'learning_rate': [0.1, 0.2, 0.4, 0.5,
                                                          1.0],
                                        'loss': ['linear', 'square',
                                                 'exponential'],
                                        'n_estimators': [10, 20, 40, 50, 100],
                                        'random_state': [7, 21, 42]})

In [23]:
ada_grid.best_params_

{'learning_rate': 0.2,
 'loss': 'exponential',
 'n_estimators': 40,
 'random_state': 7}

In [24]:
ada_random.best_params_

{'random_state': 21,
 'n_estimators': 20,
 'loss': 'exponential',
 'learning_rate': 0.4}

In [25]:
ada = AdaBoostRegressor(n_estimators=40,learning_rate=0.2,loss='exponential',random_state=7)
ada_rand = AdaBoostRegressor(n_estimators=20,learning_rate=0.4,loss='square',random_state=7)

In [26]:
models = [ada,ada_rand]
for i in models:
    i.fit(X_train,y_train)
    y_pred = i.predict(X_test)
    mae,mse,r2 = evaluate_model(y_test,y_pred)
    print(f"Model: {i}")
    print(f"Mean absolute error: {mae}")
    print(f"Mean squared error: {mse}")
    print(f"R2 score: {r2}")
    print('------------------------------------')

Model: AdaBoostRegressor(learning_rate=0.2, loss='exponential', n_estimators=40,
                  random_state=7)
Mean absolute error: 242106.82075520695
Mean squared error: 180589720769.15268
R2 score: 0.7601034947282233
------------------------------------
Model: AdaBoostRegressor(learning_rate=0.4, loss='square', n_estimators=20,
                  random_state=7)
Mean absolute error: 278948.83052724187
Mean squared error: 191631671042.04834
R2 score: 0.7454352994922467
------------------------------------
